# GNSS Paper 1 - SVM-only retrain + downstream rerun

Retrains **only the SVM detector**, this time with proper GridSearchCV hyperparameter tuning (`C`, `gamma`) -- the original run trained SVM at its config default because it was missing from `HYPERPARAM_GRIDS`, the only one of the 14 detectors that never got tuned. Then reruns every downstream stage so every table/figure reflects the tuned SVM:

`01b_retrain_svm_only -> 12_operating_point -> 13_blackbox_attacks -> 25_fragility_ci -> 14_multisurrogate_transfer -> 18_full_eval -> 17_latency -> 20_mono -> 19_final_analysis -> 22_mcnemar`

The other **13 detectors** (7 other classical + 6 deep learning, including the finalized Transformer) are **reused as-is** from a prior full run -- not retrained. That prior run's `results/models/` is a **required input** for this notebook (see Step 1b below); this saves the ~4-5h of GridSearchCV over the other 17 already-correctly-tuned classical variants and the ~1-2h of DL training that a full pipeline rerun would otherwise repeat for no reason.

This does **not** regenerate the manuscript figures. That is a separate, later step: run `papers/paper1-satnav/make_figures.py` locally against the downloaded tables.

## Before you commit, set in the right panel
Accelerator = **GPU**, Internet = **On**, and **two** Add Inputs:
1. Your dataset with `texbat_track_combined.csv` (same one used for the full pipeline runs).
2. A **new** dataset containing the prior run's trained models -- see Step 1b.

Use **Save Version -> Save & Run All (Commit)**, not the interactive Run, so the result survives if the browser tab closes.

## Step 1a. Clone the code (Paper-1 branch)

`Master_Thesis_Part_A` is a **private** repo, so an anonymous clone fails with
`could not read Username for 'https://github.com'`. Before running the next
cell, attach your GitHub token as a Kaggle Secret (NOT pasted into a cell --
a notebook cell can end up shared or public, a Secret cannot):

1. This notebook's right-hand panel -> **Add-ons -> Secrets**.
2. **Add a new secret**: label it `GITHUB_TOKEN`, value = your Personal Access
   Token (fine-grained, scoped to just this repo, `Contents: Read-only` is
   enough to clone).
3. Toggle it **Attached** for this notebook, then save.

The next cell reads it via Kaggle's secrets API at runtime; the token itself
never appears in the notebook source.

In [ ]:
import os, subprocess, sys

REPO_HOST = "github.com/Ojerinde/Master_Thesis_Part_A.git"
BRANCH    = "paper1-experiment"
DST       = "/kaggle/working/repo"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = f"https://{token}@{REPO_HOST}"
    print("Using GITHUB_TOKEN from Kaggle Secrets.")
except Exception as e:
    REPO = f"https://{REPO_HOST}"
    print(f"[WARN] No GITHUB_TOKEN secret found ({e}). Trying an anonymous clone, "
          f"which will fail with 'could not read Username' on a private repo. "
          f"See the markdown cell above to attach one.")

if not os.path.exists(DST):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, DST], check=True)
os.chdir(DST)
print("cwd:", os.getcwd()); print("top-level:", sorted(os.listdir("."))[:20])

## Step 1b. Place the corpus CSV where the loader expects it

In [ ]:
import glob, shutil, os
src = glob.glob("/kaggle/input/**/texbat_track_combined.csv", recursive=True)
assert src, "Attach the Kaggle dataset that contains texbat_track_combined.csv (right panel > Add Input)."
os.makedirs("data/processed", exist_ok=True)
shutil.copy(src[0], "data/processed/texbat_track_combined.csv")
print(f"CSV placed: {os.path.getsize('data/processed/texbat_track_combined.csv'):,} bytes")

## Step 1c. Place the prior run's OTHER 13 trained models

This is the step that makes this notebook a *targeted* rerun instead of a full retrain. Before running the next cell, upload the prior run's models as a **new Kaggle Dataset**:

1. Locally, you already have the unzipped prior run (e.g. `results_minmax_bn_tok/`, downloaded from the core-pipeline notebook's Output tab). Inside it: `results/models/classical/*.joblib` and `results/models/deep_learning/*.pt`.
2. On kaggle.com -> **Datasets -> New Dataset**, upload that `models` folder (or the whole `results_minmax_bn_tok.zip` -- the cell below searches recursively either way, Kaggle auto-extracts uploaded zips). Name it something recognizable, e.g. `paper1-bn-tok-models`.
3. Back in this notebook's right panel -> **Add Input**, search for that dataset, attach it.

The cell below copies the 7 other classical `.joblib` files and all 6 deep-learning `.pt` files into place by filename (wherever they land under `/kaggle/input/`, regardless of folder nesting) and fails loudly, listing exactly what's missing, if anything isn't found -- it does **not** silently continue with a partial set.

In [ ]:
import glob, shutil, os

os.makedirs("results/models/classical", exist_ok=True)
os.makedirs("results/models/deep_learning", exist_ok=True)

# The 7 OTHER classical stems + 6 DL stems every downstream stage expects
# (SVM itself is the 8th classical detector -- produced fresh by 01b below,
# not copied in here). Kept in sync with run_pipeline_svm_retrain.py.
REQUIRED_CLASSICAL = ['RandomForest_default', 'XGBoost_default', 'LightGBM_default',
                       'GradientBoosting', 'KNN', 'MLP', 'DecisionTree']
REQUIRED_DL = ['cnn_1d', 'lstm', 'bilstm', 'cnn_lstm', 'transformer', 'tcn']

copied, missing = [], []
for stem in REQUIRED_CLASSICAL:
    hits = glob.glob(f"/kaggle/input/**/{stem}.joblib", recursive=True)
    if hits:
        shutil.copy(hits[0], f"results/models/classical/{stem}.joblib")
        copied.append(stem + ".joblib")
    else:
        missing.append(stem + ".joblib")
for stem in REQUIRED_DL:
    hits = glob.glob(f"/kaggle/input/**/{stem}.pt", recursive=True)
    if hits:
        shutil.copy(hits[0], f"results/models/deep_learning/{stem}.pt")
        copied.append(stem + ".pt")
    else:
        missing.append(stem + ".pt")

print(f"Copied {len(copied)}/{len(REQUIRED_CLASSICAL) + len(REQUIRED_DL)}: {copied}")
assert not missing, (
    f"Missing {missing} -- attach the Kaggle dataset containing the prior run's "
    f"results/models/ folder (right panel > Add Input). See the markdown cell "
    f"above for exactly what to upload.")
print("All 13 prior-run models placed -- ready for the SVM-only retrain.")

## Step 2. Environment check (do NOT `pip install -r requirements.txt` here)

In [ ]:
import sys, subprocess, importlib, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY"))
if not torch.cuda.is_available():
    print("[WARN] No GPU detected. Set Accelerator = GPU in the right panel, then re-run.")
try:
    import cuml
    print("cuML available:", cuml.__version__, "-> SVM will tune+train with the GPU, exact RBF kernel.")
except ImportError:
    print("cuML NOT available -> SVM will use the CPU random-Fourier-features approximation "
          "(and GridSearchCV tuning will not apply to it -- see 01b's console output).")
for mod, pip_name in [("imblearn","imbalanced-learn"),("xgboost","xgboost"),("lightgbm","lightgbm"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=True)
print("deps OK")

## Step 3. Run the SVM-only retrain + downstream rerun

`run_pipeline_svm_retrain.py` preflight-checks that all 13 other-detector files landed in Step 1c, retrains+tunes only SVM, then reruns the 9 downstream attack/stats stages. Should be meaningfully faster than the full pipeline: it skips stage 01's GridSearchCV over the other 17 already-correctly-tuned classical variants and skips stage 02's 6-architecture x 3-seed DL training entirely -- the remaining cost is SVM's own tuning (a 3x3 grid x 5-fold CV, GPU-accelerated if cuML is present) plus whatever share of the original run the attack/stats stages themselves took (their cost doesn't depend on retraining -- they always reload and re-evaluate all 14 models either way).

Streams every stage's output live so an error surfaces immediately, AND tees it to `/kaggle/working/pipeline_log_svm_retrain.txt` -- the Friedman/McNemar statistics are only ever printed to the console, never written to a CSV, so this log is the only durable record of them. Fails fast: stops at the first stage that returns nonzero, including the preflight check itself if Step 1c didn't place all 13 files.

In [ ]:
import os, subprocess, sys, time
env = dict(os.environ, PYTHONPATH=".", PYTHONWARNINGS="ignore", PYTHONUTF8="1")
cmd = [sys.executable, "-u", "run_pipeline_svm_retrain.py"]
print("running:", " ".join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1, encoding="utf-8")
with open("/kaggle/working/pipeline_log_svm_retrain.txt", "w", encoding="utf-8") as logf:
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
p.wait()
print(f"[pipeline done] exit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")
assert p.returncode == 0, f"run_pipeline_svm_retrain.py failed with exit {p.returncode} (scroll up for the traceback)."

## Step 4. Package everything for download (models + tables)

Zips the full `results/` tree -- the freshly-tuned `SVM.joblib` plus the 13 reused detectors, and every result table with the SVM row now reflecting real tuning.

In [ ]:
import shutil, os
shutil.make_archive("/kaggle/working/results_svm_retrain", "zip", "results")
sz = os.path.getsize("/kaggle/working/results_svm_retrain.zip")
print(f"Wrote /kaggle/working/results_svm_retrain.zip ({sz/1e6:.1f} MB) -- contains results/models and results/tables.")

In [ ]:
import pandas as pd, os
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)
HEADLINE = [
    'results/tables/operating_point_recall95.csv',
    'results/tables/adversarial_full_oppoint.csv',
    'results/tables/blackbox_boundary_all.csv',
    'results/tables/blackbox_boundary_ci.csv',
]
for name in HEADLINE:
    if not os.path.exists(name):
        print(f'[missing] {name}'); continue
    g = pd.read_csv(name)
    print('='*70); print(f'{name}  rows={len(g)}'); print('='*70)
    col = 'model' if 'model' in g.columns else g.columns[0]
    print(g.round(4).to_string(index=False)[:4000])
    print(f'----BEGIN {os.path.basename(name)}----'); print(g.to_csv(index=False)); print(f'----END {os.path.basename(name)}----')
print()
print('SVM row specifically (the variable under test this run):')
oppoint = pd.read_csv('results/tables/operating_point_recall95.csv')
print(oppoint[oppoint.model == 'SVM'].to_string(index=False))
print()
baseline = pd.read_csv('results/tables/baseline_results.csv')
svm_row = baseline[baseline.model_name == 'SVM']
if not svm_row.empty:
    print('SVM tuning outcome (from baseline_results.csv):')
    print(svm_row[['model_name', 'hyperparameter_tuned', 'best_params', 'f1', 'auc_roc']].to_string(index=False))

### After it finishes
Download from the committed version's **Output** tab and send back:
- `results_svm_retrain.zip` -- unzip into a folder clearly labeled, e.g. `results_minmax_svm_retrain`. Contains the same `results/models/` + `results/tables/` layout as the core-pipeline notebooks, now with a properly-tuned SVM and every downstream number recomputed against it.
- `pipeline_log_svm_retrain.txt` -- full console log, including whether SVM tuning actually ran on cuML ("Best params: ...") or fell back, and the Friedman/McNemar statistics.

This becomes the final `results/` for the manuscript -- regenerate figures/numbers from this, not from the original `results_minmax_bn_tok`.